# ツール共有のためのMCP統合

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/)を使用してアプリケーション間でツールを共有します。このノートブックでは、ツールをMCPサーバーに変換し、本番環境で使用するためにデプロイする方法を説明します。

## 学習内容

- Model Context Protocolアーキテクチャを理解する
- カスタムツールをMCPサーバーに変換する
- ツール共有のためにエージェントをMCPサーバーに接続する
- 本番環境用にMCPサーバーをデプロイする

## 前提条件

- [ノートブック02: カスタムツール](02-custom-tools.ipynb)を完了していること
- HTTPサーバーとクライアント-サーバーアーキテクチャの理解
- Python Web開発の基礎

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool

print("✅ Imports successful!")

## MCPアーキテクチャを理解する

MCPは以下で構成されます：
1. **MCPサーバー**: ツールとリソースを公開する
2. **MCPクライアント**: サーバーに接続してツールを使用する
3. **プロトコル**: 標準化された通信形式

Strands Agentsは、MCPサーバーからツールを使用するMCPクライアントとして機能できます。

## MCPサーバートランスポートタイプ

Strands Agentsは**MCPサーバーを作成する4つの方法**をサポートしており、それぞれ異なるトランスポートメカニズムを持っています：

### 1. 📟 標準I/O (stdio)
**通信**: プロセスのstdin/stdoutストリーム

**使用例:**
- コマンドラインツール
- ローカル開発
- プロセス間通信

**例:**
```python
# サーバー
mcp.run(transport="stdio")

# クライアント
from mcp.client.stdio import stdio_client
client = MCPClient(lambda: stdio_client("python", "server.py"))
```

**利点:** シンプル、ネットワーク設定不要、セキュア（ローカルのみ）

**欠点:** 単一プロセス、分散システムには適さない

---

### 2. 🌊 ストリーミング可能なHTTP
**通信**: ストリーミング応答付きHTTP

**使用例:**
- リアルタイムデータストリーミング
- 長時間実行操作
- 段階的な結果

**例:**
```python
# サーバー
mcp.run(transport="http", streaming=True)

# クライアント
from mcp.client.http import http_client
client = MCPClient(lambda: http_client("http://localhost:8000"))
```

**利点:** リアルタイム更新、大きな応答に効率的

**欠点:** SSEより複雑、ストリーミングサポートが必要

---

### 3. 📡 Server-Sent Events (SSE)
**通信**: サーバー送信イベント付きHTTP

**使用例:**
- Webベースの統合
- リモートツールサーバー
- 本番環境へのデプロイ
- **これが例で使用しているものです！**

**例:**
```python
# サーバー
mcp.run(transport="sse")  # デフォルトポート: 8000

# クライアント
from mcp.client.sse import sse_client
client = MCPClient(lambda: sse_client("http://localhost:8000/sse"))
```

**利点:** HTTPベース、ファイアウォール対応、デプロイが簡単

**欠点:** 一方向通信（サーバーからクライアントへ）

---

### 4. 🔧 MCPクライアントを使用したカスタムトランスポート
**通信**: 独自のトランスポート実装

**使用例:**
- カスタムプロトコル（WebSocket、gRPCなど）
- 特別なセキュリティ要件
- 既存システムとの統合

**例:**
```python
# カスタムトランスポートを実装
class CustomTransport:
    async def connect(self): ...
    async def send(self, message): ...
    async def receive(self): ...

# MCPClientで使用
client = MCPClient(lambda: CustomTransport())
```

**利点:** 最大限の柔軟性、カスタムプロトコル

**欠点:** カスタム実装が必要

---

### トランスポート比較

| トランスポート | ネットワーク | ストリーミング | 複雑さ | 最適な用途 |
|-----------|---------|-----------|------------|----------|
| **stdio** | ❌ ローカルのみ | ❌ | 低 | CLIツール、ローカル開発 |
| **HTTP** | ✅ リモート | ✅ | 中 | リアルタイムストリーミング |
| **SSE** | ✅ リモート | ✅ | 低 | Webアプリ、本番環境 |
| **Custom** | ✅ 柔軟 | ✅ | 高 | 特別な要件 |

### 📝 このノートブックでは

**2つのトランスポートタイプ**を実演します：
1. **SSE (Server-Sent Events)** - HTTPベースのMCPサーバー用
2. **stdio (Standard I/O)** - コマンドライン使用用（オプション）

両方の例は同じツールを使用し、トランスポートメカニズムのみが異なります！

## カスタムツールからMCPサーバーを作成する

ノートブック02のカスタムツールを**2つの異なるトランスポート**を使用してMCPサーバーに変換しましょう！

サーバーは以下を公開します：
- 🧮 **calculator**: 基本的な数学演算
- ⏰ **get_current_time**: 現在の日時
- 🎥 **video_reader_local**: Bedrockを使用した動画分析

### 例1: ストリーミング可能なHTTPトランスポート

**トランスポートタイプ:** 📡 ストリーミング可能なHTTP

**最適な用途:** Webベースの統合、リモートアクセス、本番環境へのデプロイ

FastMCPを使用してストリーミング可能なHTTPトランスポートで`mcp_custom_tools_server.py`を作成します：

In [ ]:
# MCPサーバーファイルを作成する
mcp_server_code = '''#!/usr/bin/env python3
"""
カスタムツール用MCPサーバー
このサーバーはModel Context Protocol経由で計算機、時刻、動画分析ツールを公開します。

Strands AgentsエコシステムのFastMCPで構築されています。
"""
import os
from datetime import datetime
from mcp.server import FastMCP

# 動画リーダーツールをインポート
from video_reader_local import video_reader_local

# FastMCPサーバーを作成
mcp = FastMCP("Custom Tools Server")


@mcp.tool(description="基本的な数学演算を実行します（加算、減算、乗算、除算）")
def calculator(operation: str, a: float, b: float) -> str:
    """
    基本的な数学演算用の計算機ツール。
    
    Args:
        operation: 実行する演算（add, subtract, multiply, divide）
        a: 最初の数値
        b: 2番目の数値
    
    Returns:
        演算結果を文字列として返す
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    
    result = operations.get(operation, "Invalid operation")
    return str(result)


@mcp.tool(description="UTCタイムゾーンで現在の日時を取得します")
def get_current_time(timezone: str = "UTC") -> str:
    """
    現在の日時を取得します。
    
    Args:
        timezone: タイムゾーン（現在はUTCのみサポート）
    
    Returns:
        フォーマットされた文字列として現在の日時を返す
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


@mcp.tool(description="""AWS Bedrockのマルチモーダル機能を使用して動画コンテンツを分析します。

重要な制限事項:
- リクエストごとに1つの動画のみ
- 音声分析なし（視覚のみ）
- 人物の識別不可
- 最大ファイルサイズ: ~20MB
- サポート形式: mp4, mov, avi, mkv, webm
""")
def analyze_video(
    video_path: str,
    text_prompt: str = "この動画で見えるものを説明してください",
    model_id: str = "us.amazon.nova-pro-v1:0",
    region: str = "us-west-2"
) -> str:
    """
    AWS Bedrockを使用して動画コンテンツを分析します。
    
    Args:
        video_path: ローカル動画ファイルのパス
        text_prompt: 動画を分析するための質問または指示
        model_id: 使用するBedrockモデルID
        region: BedrockのAWSリージョン
    
    Returns:
        動画分析結果を文字列として返す
    """
    # 動画リーダーツールを呼び出す
    result = video_reader_local(
        video_path=video_path,
        text_prompt=text_prompt,
        model_id=model_id,
        region=region
    )
    
    # 結果からテキストを抽出
    return result["content"][0]["text"]


# サーバーを実行
if __name__ == "__main__":
    mcp.run(transport="streamable-http") # リモート通信にストリーミング可能なHTTPを使用
'''

# サーバーファイルを書き込む
with open('mcp_custom_tools_server.py', 'w') as f:
    f.write(mcp_server_code)

print("✅ MCPサーバーファイルを作成しました: mcp_custom_tools_server.py")
print("\n📝 MCPサーバーの主要コンポーネント（ストリーミング可能なHTTPトランスポート）:")
print("="*80)
print("1. from mcp.server import FastMCP")
print("2. mcp = FastMCP('Custom Tools Server')")
print("3. @mcp.tool() - ツールを定義するデコレータ")
print("4. mcp.run(transport='streamable-http') - ストリーミング可能なHTTPサーバーを起動")
print("="*80)
print("\n🌐 ストリーミング可能なHTTP MCPサーバーは以下で実行されます: http://localhost:8000/mcp")
print("📡 トランスポート: ストリーミング可能なHTTP")
print("✅ 最適な用途: Web統合、リモートアクセス、本番環境")

### 例2: SSEトランスポート

**トランスポートタイプ:** 📡 SSE HTTP

**最適な用途:** Webベースの統合、リモートアクセス、本番環境へのデプロイ

FastMCPを使用してSSE HTTPトランスポートで`mcp_custom_tools_server_sse.py`を作成します：

In [ ]:
# MCPサーバーファイルを作成する
mcp_server_code = '''#!/usr/bin/env python3
"""
カスタムツール用MCPサーバー
このサーバーはModel Context Protocol経由で計算機、時刻、動画分析ツールを公開します。

Strands AgentsエコシステムのFastMCPで構築されています。
"""
import os
from datetime import datetime
from mcp.server import FastMCP

# 動画リーダーツールをインポート
from video_reader_local import video_reader_local

# FastMCPサーバーを作成
mcp = FastMCP("Custom Tools Server")


@mcp.tool(description="基本的な数学演算を実行します（加算、減算、乗算、除算）")
def calculator(operation: str, a: float, b: float) -> str:
    """
    基本的な数学演算用の計算機ツール。
    
    Args:
        operation: 実行する演算（add, subtract, multiply, divide）
        a: 最初の数値
        b: 2番目の数値
    
    Returns:
        演算結果を文字列として返す
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    
    result = operations.get(operation, "Invalid operation")
    return str(result)


@mcp.tool(description="UTCタイムゾーンで現在の日時を取得します")
def get_current_time(timezone: str = "UTC") -> str:
    """
    現在の日時を取得します。
    
    Args:
        timezone: タイムゾーン（現在はUTCのみサポート）
    
    Returns:
        フォーマットされた文字列として現在の日時を返す
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


@mcp.tool(description="""AWS Bedrockのマルチモーダル機能を使用して動画コンテンツを分析します。

重要な制限事項:
- リクエストごとに1つの動画のみ
- 音声分析なし（視覚のみ）
- 人物の識別不可
- 最大ファイルサイズ: ~20MB
- サポート形式: mp4, mov, avi, mkv, webm
""")
def analyze_video(
    video_path: str,
    text_prompt: str = "この動画で見えるものを説明してください",
    model_id: str = "us.amazon.nova-pro-v1:0",
    region: str = "us-west-2"
) -> str:
    """
    AWS Bedrockを使用して動画コンテンツを分析します。
    
    Args:
        video_path: ローカル動画ファイルのパス
        text_prompt: 動画を分析するための質問または指示
        model_id: 使用するBedrockモデルID
        region: BedrockのAWSリージョン
    
    Returns:
        動画分析結果を文字列として返す
    """
    # 動画リーダーツールを呼び出す
    result = video_reader_local(
        video_path=video_path,
        text_prompt=text_prompt,
        model_id=model_id,
        region=region
    )
    
    # 結果からテキストを抽出
    return result["content"][0]["text"]


# サーバーを実行
if __name__ == "__main__":
    mcp.run(transport="sse") # リモート通信にSSEを使用
'''

# サーバーファイルを書き込む
with open('mcp_custom_tools_server_sse.py', 'w') as f:
    f.write(mcp_server_code)

print("✅ MCPサーバーファイルを作成しました: mcp_custom_tools_server_sse.py")
print("\n📝 MCPサーバーの主要コンポーネント（SSEトランスポート）:")
print("="*80)
print("1. from mcp.server import FastMCP")
print("2. mcp = FastMCP('Custom Tools Server')")
print("3. @mcp.tool() - ツールを定義するデコレータ")
print("4. mcp.run(transport='sse') - SSE HTTPサーバーを起動")
print("="*80)
print("\n🌐 SSEサーバーは以下で実行されます: http://localhost:8000/sse")
print("📡 トランスポート: Server-Sent Events (SSE)")
print("✅ 最適な用途: Web統合、リモートアクセス、本番環境")

### 例3: stdioトランスポート（標準I/O）

**トランスポートタイプ:** 📟 標準I/O (stdio)

**最適な用途:** コマンドラインツール、ローカル開発、プロセス間通信

stdioトランスポートを使用した代替バージョンを作成しましょう：

In [ ]:
# stdioバージョンのMCPサーバーを作成する
stdio_server_code = '''#!/usr/bin/env python3
"""
カスタムツール用MCPサーバー - stdioトランスポート
このサーバーはプロセス間通信に標準I/Oを使用します。

Strands AgentsエコシステムのFastMCPで構築されています。
"""
import os
from datetime import datetime
from mcp.server import FastMCP

# 動画リーダーツールをインポート
from video_reader_local import video_reader_local

# FastMCPサーバーを作成
mcp = FastMCP("Custom Tools Server - stdio")


@mcp.tool(description="基本的な数学演算を実行します（加算、減算、乗算、除算）")
def calculator(operation: str, a: float, b: float) -> str:
    """基本的な数学演算用の計算機ツール。"""
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    result = operations.get(operation, "Invalid operation")
    return str(result)


@mcp.tool(description="UTCタイムゾーンで現在の日時を取得します")
def get_current_time(timezone: str = "UTC") -> str:
    """現在の日時を取得します。"""
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


@mcp.tool(description="AWS Bedrockを使用して動画コンテンツを分析します")
def analyze_video(
    video_path: str,
    text_prompt: str = "この動画で見えるものを説明してください",
    model_id: str = "us.amazon.nova-pro-v1:0",
    region: str = "us-west-2"
) -> str:
    """AWS Bedrockを使用して動画コンテンツを分析します。"""
    result = video_reader_local(
        video_path=video_path,
        text_prompt=text_prompt,
        model_id=model_id,
        region=region
    )
    
    if result["status"] == "success":
        return result["content"][0]["text"]
    else:
        return result["content"][0]["text"]


# stdioトランスポートでサーバーを実行
if __name__ == "__main__":
    mcp.run(transport="stdio")  # コマンドライン通信にstdioを使用
'''

# stdioサーバーファイルを書き込む
with open('mcp_custom_tools_server_stdio.py', 'w') as f:
    f.write(stdio_server_code)

print("✅ stdio MCPサーバーファイルを作成しました: mcp_custom_tools_server_stdio.py")
print("\n📝 主な違い（stdioトランスポート）:")
print("="*80)
print("1. mcp.run(transport='stdio') - stdin/stdoutを使用")
print("2. HTTPサーバーなし - 直接プロセス通信")
print("3. 起動方法: python mcp_custom_tools_server_stdio.py")
print("="*80)
print("\n📟 トランスポート: 標準I/O (stdio)")
print("✅ 最適な用途: CLIツール、ローカル開発、プロセス間通信")
print("\n💡 両方のサーバーは同じツールを公開し、トランスポートのみが異なります！")

### トランスポート比較: SSE vs stdio

| 機能 | SSE (mcp_custom_tools_server_sse.py) | stdio (mcp_custom_tools_server_stdio.py) |
|---------|----------------------------------|------------------------------------------|
| **トランスポート** | Server-Sent Events付きHTTP | 標準I/Oストリーム |
| **ネットワーク** | ✅ リモートアクセス | ❌ ローカルのみ |
| **URL** | http://localhost:8000/sse | N/A（プロセス通信） |
| **クライアント** | `sse_client(url)` | `stdio_client(command, args)` |
| **使用例** | Webアプリ、本番環境 | CLIツール、ローカル開発 |
| **ファイアウォール** | 設定が必要な場合あり | ネットワーク不要 |
| **デプロイ** | Docker、クラウドサービス | ローカルプロセス |

### どのトランスポートを選ぶべきか？

**SSEを使用する場合:**
- Webアプリケーションを構築する場合
- ツールへのリモートアクセスが必要な場合
- クラウド（AWS、Docker）にデプロイする場合
- 複数のクライアントがアクセスする必要がある場合

**stdioを使用する場合:**
- CLIツールを構築する場合
- ローカル開発のみの場合
- プロセス間通信の場合
- ネットワークアクセスが不要な場合

**このノートブックでは、SSEを使用します**。本番シナリオとWebベースの統合により適しているためです。

## Strands AgentsでMCPサーバーを使用する

**ストリーミング可能なHTTPベースのMCPサーバー**に接続し、Strands Agentsで使用しましょう。

### ステップ1: ストリーミング可能なHTTP MCPサーバーを起動する

**別のターミナル**を開いて実行します：

```bash
python mcp_custom_tools_server.py
```

サーバーはデフォルトで`http://localhost:8000/mcp/`で起動します。

**注意:** stdioバージョンを使用する場合は、異なる方法で起動します（以下の手動アプローチを参照）。

### ステップ2: MCPサーバーに接続する

#### オプションA: ストリーミング可能なHTTP MCPサーバーに接続する


In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

# SSE経由でカスタムツールMCPサーバーに接続する
mcp_client = MCPClient(
    lambda: streamablehttp_client("http://localhost:8000/mcp/")
)

print("✅ MCPクライアントを作成しました（ストリーミング可能なHTTPトランスポート）")
print("📡 トランスポート: ストリーミング可能なHTTP")
print("🌐 接続先: http://localhost:8000/mcp/")
print("⚠️  別のターミナルでmcp_custom_tools_server.pyが実行されていることを確認してください！")
print("\n💡 代替案: stdioトランスポートを使用する場合:")
print("   from mcp.client.stdio import stdio_client")

#### オプションB: SSE MCPサーバーに接続する:
代替として、SSEトランスポートプロトコルを使用してMCPサーバーに接続します。

In [ ]:
from mcp.client.sse import sse_client
from strands.tools.mcp import MCPClient

# SSE経由でカスタムツールMCPサーバーに接続する
mcp_client = MCPClient(
    lambda: sse_client("http://localhost:8000/sse")
)

print("✅ MCPクライアントを作成しました（SSEトランスポート）")
print("📡 トランスポート: Server-Sent Events (SSE)")
print("🌐 接続先: http://localhost:8000/sse")
print("⚠️  別のターミナルでmcp_custom_tools_server_sse.pyが実行されていることを確認してください！")
print("\n💡 代替案: stdioトランスポートを使用する場合:")
print("   from mcp.client.stdio import stdio_client")
print("   client = MCPClient(lambda: stdio_client('python', 'mcp_custom_tools_server_stdio.py'))")

### ステップ3: MCPツールでエージェントを作成する

Strands Agentsは、MCP統合のために2つのアプローチをサポートしています：

#### オプションA: マネージド統合（推奨 - 実験的）
エージェントがMCP接続のライフサイクルを自動的に管理します：

In [ ]:
# Bedrockモデルを設定する
session = boto3.Session(region_name='us-west-2')
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    boto_session=session
)

# MCPクライアントを直接使用してエージェントを作成する（マネージドライフサイクル）
mcp_agent = Agent(
    model=bedrock_model,
    tools=[mcp_client],  # MCPClientを直接渡す！
    system_prompt="""あなたは次の機能にアクセスできる親切なアシスタントです：
    - 数学演算用の計算機
    - 時刻情報
    - 動画分析機能
    """
)

print("✅ MCPツールでエージェントを作成しました（マネージドモード）！")
print("📦 エージェントは自動的に接続してツールを検出します")

#### オプションB: 手動コンテキスト管理
本番環境では、接続を手動で管理できます：

In [ ]:
# 明示的なコンテキスト管理による手動アプローチ
# このアプローチを使用するには、コメントを外してください：

with mcp_client:
    # MCPサーバーからツールを取得する
    mcp_tools = mcp_client.list_tools_sync()
    
    # ツールでエージェントを作成する
    mcp_agent = Agent(
        model=bedrock_model,
        tools=mcp_tools,
        system_prompt="あなたは親切なアシスタントです..."
    )
    
    # コンテキスト内でエージェントを使用する
    response = mcp_agent("10 + 20は何ですか？")
    print(response)

print("ℹ️  手動アプローチはコメントアウトされています - 上記のマネージドモードを使用しています")

### ステップ4: MCPツールをテストする

In [ ]:
# テスト1: MCP経由の計算機
print("=== 🧮 計算機テスト ===")
response = mcp_agent("456に789を掛けると何ですか？")
# print(response)
print("\n" + "="*80 + "\n")

In [ ]:
# テスト2: MCP経由の時刻
print("=== ⏰ 時刻テスト ===")
response = mcp_agent("今何時ですか？")
# print(response)
print("\n" + "="*80 + "\n")

In [ ]:
# # テスト3: MCP経由の動画分析
# print("=== 🎥 動画分析テスト ===")
# response = mcp_agent(
#     "data-sample/moderation-video.mp4の動画を分析して、見えるものを説明してください"
# )
# print(response)
# print("\n" + "="*80 + "\n")

In [ ]:
# テスト4: 1つのリクエストで複数のツールを使用
print("=== 🔧 マルチツールテスト ===")
response = mcp_agent(
    "今何時ですか？また、123 + 456を計算してください。"
)
# print(response)
print("\n" + "="*80 + "\n")

## MCPサーバーでマルチモーダルエージェントをテストする

複数のMCPツールを組み合わせた実用的な例を見てみましょう：

In [ ]:
# 実世界のシナリオ: タイムスタンプ付き動画コンテンツモデレーション
print("=== 🎯 実用例: コンテンツモデレーション ===")

response = mcp_agent("""
コンテンツモデレーションをお願いします：
1. まず、この分析が実行されている時刻を教えてください
2. 次に、data-sample/moderation-video.mp4の動画を分析してください
3. 気になるコンテンツが見つかった場合、0-100のリスクスコアを計算してください
""")

print(response)
print("\n" + "="*80)

## StrandsツールをMCPサーバーに変換する

**FastMCP**を使用してノートブック02のツールをMCPサーバーに変換した方法を説明します：

### 元のStrandsツール:

In [ ]:
# 元のStrandsツール（ノートブック02から）
@tool
def calculator(operation: str, a: float, b: float) -> float:
    """基本的な数学演算を実行します。"""
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Invalid operation")

print("✅ 元のStrandsツール")

### FastMCPでMCPサーバーに変換:

**オプション1: SSEトランスポート（HTTPベース）**
```python
# mcp_custom_tools_server_sse.py内

from mcp.server import FastMCP

# FastMCPサーバーを作成
mcp = FastMCP("Custom Tools Server")

# デコレータでツールを定義
@mcp.tool(description="基本的な数学演算を実行します")
def calculator(operation: str, a: float, b: float) -> str:
    """基本的な数学演算用の計算機ツール。"""
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    result = operations.get(operation, "Invalid operation")
    return str(result)

# SSEトランスポートで実行
if __name__ == "__main__":
    mcp.run(transport="sse")  # 📡 HTTPアクセス用のSSEトランスポート
```

**オプション2: stdioトランスポート（プロセスベース）**
```python
# mcp_custom_tools_server_stdio.py内

# 同じツール定義...

# stdioトランスポートで実行
if __name__ == "__main__":
    mcp.run(transport="stdio")  # 📟 CLI用のstdioトランスポート
```

### Strandsツールとの主な違い:
1. **FastMCPのシンプルさ**: 生のMCPよりはるかにシンプル - `@mcp.tool()`デコレータを使用するだけ
2. **型ヒント**: FastMCPはPythonの型ヒントから自動的にJSONスキーマを生成
3. **トランスポートの柔軟性**: 4つのトランスポートタイプ（stdio、SSE、HTTP、カスタム）から選択可能
4. **プロセス分離**: 別プロセスとして実行（HTTPサーバーまたはCLI）
5. **同じロジック**: コアツールロジックは同一のまま！
6. **再利用性**: 1つのサーバーに複数のクライアントが接続可能

### トランスポート選択ガイド:

```python
# 📟 stdio - ローカルCLIツール
mcp.run(transport="stdio")

# 📡 SSE - Webアプリ、本番環境（選択した方法）
mcp.run(transport="sse")

# 🌊 HTTPストリーミング - リアルタイムデータ
mcp.run(transport="http", streaming=True)

# 🔧 カスタム - 独自のプロトコル
mcp.run(transport=CustomTransport())
```

## MCP統合の利点

### 1. ツールの再利用性
一度ツールを作成すれば、異なるAIアプリケーション間で使用可能

### 2. 標準化
ツール統合の業界標準に従う

### 3. 関心の分離
- ツールは別プロセスとして実行される
- セキュリティと分離の向上
- 独立したスケーリング

### 4. エコシステム
成長するMCPサーバーのエコシステムにアクセス可能

## MCP計算機の例

![MCP Calculator Diagram](image/mcp_calculator_diagram.png)

図は、エージェントがMCP計算機サーバーとどのように相互作用するかを示しています：
1. エージェントがユーザーリクエストを受信
2. エージェントがMCPサーバーツールを呼び出し
3. MCPサーバーが計算を処理
4. 結果がエージェントに返される
5. エージェントがユーザーに応答

## MCPサーバーのデプロイ

MCPサーバーは以下の方法でデプロイできます：
- 🖥️ 開発用にローカルで
- 🐳 コンテナ（Docker）内で
- ☁️ AWS Lambda、ECS、EC2、Agentcore Runtime、AgentCore Gateway上で
- 🌐 スタンドアロンサービスとして


## クリーンアップ

マネージドモードを使用する場合、エージェントが自動的にクリーンアップを処理します。
手動モードの場合、MCPクライアントを閉じます：

In [ ]:
# マネージドモード: クリーンアップ不要 - エージェントが自動的に処理
print("✅ マネージドモードを使用中 - 自動クリーンアップ")

# 手動モード: 手動コンテキスト管理を使用する場合はコメントを外してください
# mcp_client.close()
# print("✅ MCPクライアントを閉じました")

## まとめ

このノートブックでは、以下を学習しました：

✅ Model Context Protocol (MCP)とは何か、なぜ重要なのか

✅ StrandsツールをMCPサーバーに変換する方法

✅ Strands AgentsからMCPサーバーに接続する方法

✅ エージェントでMCPツールを使用する方法

✅ MCP統合の実世界の例


### 次のステップ

次のノートブックに進んで、State & Sessions管理について学習しましょう！